# IPL / Cricket Player Performance Analysis (Project #21)

**Dataset:** Agent-generated synthetic IPL benchmark dataset (`2021`–`2024`, `160` matches, `32` players, `1,280` clean player-match rows).  
**Objective:** Verify data cleaning integrity, validate 5 benchmark players (`P001`–`P005`), compute core batting/bowling KPIs, inspect the 4 analytical PNG charts, and document key insights.

## 1. Setup & Data Quality Audit Check
**What & Why:** Import `pandas`, `json`, and `sqlite3`, configure project paths, and inspect `reports/data_quality_log.csv` to verify that every raw data issue was fixed transparently without silently dropping rows.

In [ ]:
import json
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import Image, display

ROOT = Path('..').resolve()
CLEAN_DIR = ROOT / 'data' / 'clean'
REPORTS_DIR = ROOT / 'reports'

dq_log = pd.read_csv(REPORTS_DIR / 'data_quality_log.csv')
dq_log

## 2. Load Cleaned Star-Schema Tables & Check Dimensions
**What & Why:** Load all 5 cleaned CSV tables with `keep_default_na=False` (preserving literal `'None'` in `bowling_style`) and verify row counts (`8` teams, `4` seasons, `32` players, `160` matches, `1,280` fact rows).

In [ ]:
teams = pd.read_csv(CLEAN_DIR / 'dim_teams.csv', na_values=[''], keep_default_na=False)
seasons = pd.read_csv(CLEAN_DIR / 'dim_seasons.csv', na_values=[''], keep_default_na=False)
players = pd.read_csv(CLEAN_DIR / 'dim_players.csv', na_values=[''], keep_default_na=False)
matches = pd.read_csv(CLEAN_DIR / 'dim_matches.csv', na_values=[''], keep_default_na=False)
facts = pd.read_csv(CLEAN_DIR / 'fact_player_match_stats.csv', na_values=[''], keep_default_na=False)

for name, df in [('dim_teams', teams), ('dim_seasons', seasons), ('dim_players', players), ('dim_matches', matches), ('fact_player_match_stats', facts)]:
    print(f'{name}: {len(df)} rows x {len(df.columns)} cols')

## 3. Verify 5 Benchmark Players & Python-vs-SQLite Parity
**What & Why:** Cross-check `reports/kpis.json` and `reports/sql_outputs/q1_5_player_verification.csv` to ensure `not_out`-adjusted Batting Average, Strike Rate, Wickets, and Economy match identically across Python and SQL.

In [ ]:
kpis = json.loads((REPORTS_DIR / 'kpis.json').read_text(encoding='utf-8'))
print('Overall KPIs:', {k: kpis[k] for k in ['total_matches', 'total_players', 'total_runs', 'total_wickets', 'overall_batting_average', 'overall_strike_rate', 'overall_economy_rate']})

q1_sql = pd.read_csv(REPORTS_DIR / 'sql_outputs' / 'q1_5_player_verification.csv')
q1_sql

## 4. Inspect Tie-Aware Season Rankings (`RANK()` & `DENSE_RANK()`)
**What & Why:** Display the season batting and bowling leaderboards exported from SQLite (`q2_season_batting_ranks.csv` and `q3_season_bowling_ranks.csv`) to confirm tied ranks in `2023` batting (`739` runs) and `2021`/`2024` bowling (`27` and `21` wickets) are preserved accurately.

In [ ]:
q2 = pd.read_csv(REPORTS_DIR / 'sql_outputs' / 'q2_season_batting_ranks.csv')
q3 = pd.read_csv(REPORTS_DIR / 'sql_outputs' / 'q3_season_bowling_ranks.csv')
print('--- 2023 Batting Tie (Rank 1) ---')
display(q2[(q2['season_year'] == 2023) & (q2['rank_skip'] <= 3)])
print('--- 2024 Bowling Tie (Rank 3) ---')
display(q3[(q3['season_year'] == 2024) & (q3['rank_skip'] <= 5)])

## 5. Analytical Visualizations
**What & Why:** Render the 4 PNG charts generated by `src/analyze_and_plot.py` covering monthly trends, franchise comparison, player-role contributions, and data quality / missing-balls impact.

In [ ]:
chart_files = [
    'monthly_runs_trend.png',
    'team_performance_comparison.png',
    'role_contribution_breakdown.png',
    'data_quality_and_missing_balls.png',
]
for cf in chart_files:
    display(Image(filename=str(REPORTS_DIR / 'charts' / cf)))

## 6. Automated Pipeline Verification Results
**What & Why:** Load `reports/test_results.csv` produced by `tests/test_pipeline.py` to confirm all 6 data integrity, parity, and benchmark checks passed.

In [ ]:
pd.read_csv(REPORTS_DIR / 'test_results.csv')

## 7. Key Insights & Recommendations

1. **Verified Aggregate Parity:** Across `160` matches and `1,280` clean player-match rows (`977` active batting innings), the pipeline recorded **`35,978` total runs** (Batting Average **`51.62`**, Strike Rate **`147.05`**) and **`1,143` wickets** (Economy **`8.21`**), matching identically between Python and SQLite.
2. **5-Player Benchmark Validation:** `P001 Virat Kohli` (`2,986` runs, Avg `66.36`, SR `146.44`), `P002 Rohit Sharma` (`2,731` runs, Avg `65.02`, SR `147.14`), `P003 Jasprit Bumrah` (`93` wickets, Econ `7.94`), `P004 Rashid Khan` (`79` wickets, Econ `8.17`), and `P005 Suryakumar Yadav` (`2,673` runs, Avg `65.20`, SR `149.83`) matched expected totals 100%.
3. **Missing Balls Sensitivity:** Recovering `25` missing `balls_bowled` values from `overs_bowled` preserved bowling economy accuracy, while imputing `15` missing `balls_faced` entries shifted overall strike rate minimally from **`146.98`** to **`147.05`**.
4. **Tie-Preserving Rankings:** `RANK()` / `DENSE_RANK()` preserved exact ties (`2023` batting Rank 1 tie at `739` runs between Virat Kohli and Suryakumar Yadav; `2024` bowling Rank 3 tie at `21` wickets between Rashid Khan and Jasprit Bumrah).
5. **Franchise, Role & Toss Recommendations:** **MI (`7,188` runs)** and **RCB (`7,165` runs)** led team scoring while **SRH (`195` wickets)** led wicket-taking. Teams electing to **field first** won **`53.76%` (`50/93`)** vs **`35.82%` (`24/67`)** when batting first—supporting a bowl-first strategy combined with All-Rounder depth (**`8,229` runs + `265` wickets**).